# 06 · Experiments and the final model (Step 7)

**Goal:** three experiments, then the final model that every later step uses.

- **A. Fix:** the "still running previous loans" features were empty in Steps 5 and 6. Fix them and measure LightGBM v1 again.
- **B. Class imbalance:** only 8% of loans default. Do class weights, undersampling or SMOTE help?
- **C. Tuning:** grid search vs random search vs Bayesian optimisation (Optuna), with the same budget. The best settings become the **final model**, saved in `models/`.

**Before running:** rebuild the features with `make features` (the setup cell checks this).

**Time:** about 1 to 1.5 hours; the tuning (Part C) is most of it.
**You can stop and re-run at any time:** model predictions are saved in `data/processed/oof/` and
every tuning trial in `data/processed/tuning/`, so a re-run continues where it stopped.

**Saved results:**

- `reports/results_step7_imbalance.csv`, `reports/results_step7_tuning.csv`
- `reports/figures/fig16_imbalance_calibration.png`, `reports/figures/fig17_tuning_curves.png`
- `models/lightgbm_final.joblib` (plus `.txt` and `.json`)
- MLflow (`make mlflow`)

In [ ]:
%load_ext autoreload
%autoreload 2

import dataclasses

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import calibration_curve

from creditrisk.config import PROJECT_ROOT, get_path, set_seed
from creditrisk.data.split import load_splits
from creditrisk.evaluation.bootstrap import paired_bootstrap
from creditrisk.evaluation.cv import load_result, results_table, run_or_load
from creditrisk.features.build import load_features
from creditrisk.models import baselines, tune
from creditrisk.models.baselines import default_threads
from creditrisk.models.data import column_groups, get_xy
from creditrisk.models.final import save_final_model
from creditrisk.models.imbalance import imbalance_models

SEED = set_seed()
FIG = get_path("figures")
REPORTS = PROJECT_ROOT / "reports"
TUNE_DIR = get_path("processed") / "tuning"
facts = {}

features = load_features()
splits = load_splits()
X, y, folds, ids = get_xy(features, splits, "train")
yv = np.asarray(y)
groups = column_groups(X)
assert X["PREV_RUNNING_CREDIT_TOTAL"].notna().any(), (
    "PREV_RUNNING_CREDIT_TOTAL is still empty: run `make features`, then restart the kernel.")
print(f"train split: {X.shape[0]:,} rows x {X.shape[1]} features | threads: {default_threads()}")

## A. Fix the "still running" previous loans

**What was wrong:** in `previous_application`, a loan that has not ended yet has the code 365243
in its end-date columns (the data has no real future dates). Step 5 turned 365243 into "missing"
first, and only then looked for end dates in the future, so it never found a running loan:
`PREV_RUNNING_COUNT` was always 0 and `PREV_RUNNING_CREDIT_TOTAL` was always empty.
Now "still running" is read **before** 365243 is removed.

Steps 5 and 6 used the old features. From here on, every model uses the fixed ones.

In [ ]:
running = (X["PREV_RUNNING_COUNT"].fillna(0) > 0).to_numpy()
rate_running, rate_not = yv[running].mean(), yv[~running].mean()
useless = [c for c in X.columns if X[c].nunique(dropna=True) <= 1]
print(f"applicants with at least one running previous loan: {running.mean():.1%}")
print(f"default rate: with a running loan {rate_running:.2%} | without {rate_not:.2%}")
print("empty or constant columns:", useless or "none")
facts["running_loan_share"] = round(float(running.mean()), 4)
facts["default_running_vs_not"] = (round(float(rate_running), 4), round(float(rate_not), 4))
facts["empty_or_constant_columns"] = useless

In [ ]:
v1 = run_or_load(lambda: baselines.lightgbm(seed=SEED), X, y, folds, "step7_lightgbm_v1_fixed",
                 ids=ids, step=7, verbose=True)
v1.name = "lightgbm_v1 (fixed features)"
try:
    v1_old = load_result("step5_ablation_5", ids=ids, display_name="lightgbm_v1 (old features)")
    gain = paired_bootstrap(yv, v1.oof, v1_old.oof)
    display(results_table([v1_old, v1]))
    print(f"fixed - old: {gain['difference']:+.4f} "
          f"(95% CI {gain['low']:+.4f} to {gain['high']:+.4f}), significant: {gain['significant']}")
    facts["v1_old_vs_fixed"] = (round(v1_old.summary["roc_auc"], 4), round(v1.summary["roc_auc"], 4))
    facts["fix_gain"] = (round(gain["difference"], 4), gain["significant"])
except FileNotFoundError:
    display(results_table([v1]))
    print("Step 5 predictions not found, so no before/after comparison.")
    facts["v1_fixed"] = round(v1.summary["roc_auc"], 4)

## B. Class imbalance

Only about 1 in 12 loans defaults. Three common "fixes", all applied **only to the training
fold** (the held-out fold is never changed):

- **Class weights:** each defaulter counts about 11 times as much in the loss.
- **Undersampling:** randomly drop non-defaulters until there is 1 defaulter for every 2 others.
- **SMOTE:** create synthetic defaulters between real neighbouring defaulters, up to the same 1:2 ratio.

SMOTE needs complete numbers, so it needs extra preprocessing (fill gaps, scale, code the text
columns). That preprocessing alone could change the score, so it gets its own **control**,
`imputed_no_smote`: the same preprocessing, without SMOTE. The real SMOTE effect is
smote vs that control. "none" is LightGBM v1 from Part A.

**What to watch:** ROC-AUC only checks the *ranking* of applicants. These methods mainly change
the *probabilities*, and Step 10 sets the approval threshold from those probabilities, so the
calibration columns (Brier, ECE, average predicted probability) matter as much as ROC-AUC.

In [ ]:
imbalance = {"none": dataclasses.replace(v1, name="none")}
for name, make_model in imbalance_models(groups, seed=SEED).items():
    if name == "none":
        continue  # the same model as LightGBM v1 above, already trained
    r = run_or_load(make_model, X, y, folds, f"step7_imb_{name}", ids=ids, step=7, verbose=True)
    r.name = name
    imbalance[name] = r

vs_none = {m: paired_bootstrap(yv, r.oof, imbalance["none"].oof)
           for m, r in imbalance.items() if m != "none"}
table = results_table(imbalance.values())
table["vs none"] = ["—" if m == "none" else
                    f"{vs_none[m]['difference']:+.4f}" + ("*" if vs_none[m]["significant"] else "")
                    for m in table.index]
table["avg predicted p"] = [round(float(imbalance[m].oof.mean()), 4) for m in table.index]
table["actual default rate"] = round(float(yv.mean()), 4)
table.to_csv(REPORTS / "results_step7_imbalance.csv")
display(table[["ROC-AUC", "95% CI", "vs none", "PR-AUC", "Brier", "ECE", "avg predicted p",
               "actual default rate"]])
print("* = significant change in ROC-AUC vs none (paired bootstrap)")

smote_effect = paired_bootstrap(yv, imbalance["smote"].oof, imbalance["imputed_no_smote"].oof)
print(f"SMOTE effect (smote - imputed_no_smote): {smote_effect['difference']:+.4f} "
      f"(95% CI {smote_effect['low']:+.4f} to {smote_effect['high']:+.4f}), "
      f"significant: {smote_effect['significant']}")

facts["imbalance_auc"] = {m: round(r.summary["roc_auc"], 4) for m, r in imbalance.items()}
facts["imbalance_vs_none"] = {m: (round(c["difference"], 4), c["significant"])
                              for m, c in vs_none.items()}
facts["imbalance_brier"] = {m: round(r.summary["brier"], 4) for m, r in imbalance.items()}
facts["imbalance_ece"] = {m: round(r.summary["ece"], 4) for m, r in imbalance.items()}
facts["imbalance_avg_p"] = {m: round(float(r.oof.mean()), 4) for m, r in imbalance.items()}
facts["smote_effect"] = (round(smote_effect["difference"], 4), smote_effect["significant"])

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
for name, r in imbalance.items():
    actual, predicted = calibration_curve(yv, r.oof, n_bins=10, strategy="quantile")
    ax1.plot(predicted, actual, marker="o", label=name)
    ax2.hist(r.oof, bins=60, range=(0, 1), histtype="step", density=True, label=name)
ax1.plot([0, 1], [0, 1], "k--", label="perfect")
ax1.set(xlabel="average predicted probability", ylabel="actual default rate",
        title="Reliability: 10 equal-size groups of applicants")
ax2.set(xlabel="predicted probability of default", ylabel="density", yscale="log",
        title="Spread of the predicted probabilities")
ax1.legend()
ax2.legend()
fig.tight_layout()
fig.savefig(FIG / "fig16_imbalance_calibration.png", dpi=150)
plt.show()

## C. Tuning: grid vs random vs Bayesian optimisation (Optuna)

All three methods get the **same budget: 27 LightGBM fits**, and every fit is checked the same
way: train on folds 2–5, keep adding trees until the ROC-AUC on fold 1 has not improved for 100
rounds (**early stopping**), and record that ROC-AUC.

- **Grid search:** every combination of 3 learning rates × 3 leaf counts × 3 minimum leaf sizes. The other settings stay at the v1 values.
- **Random search:** 27 random settings from wider ranges, for all six settings.
- **Bayesian optimisation (Optuna TPE):** the first 8 tries are random. After that, each new try is chosen using all earlier results, aiming at the most promising area.

The v1 settings are scored the same way first, as the reference line.

Each trial prints one line and is saved as soon as it finishes. You can stop at any time and run
the cell again later: finished trials are not repeated.

**Honest note:** fold 1 is used to choose the settings, so the tuned model's 5-fold score is a
little optimistic. Also, fold 1 has only about 37,000 rows, so small differences between trials
can be noise. The final, unbiased number comes from the locked test set in Step 15.

In [ ]:
BUDGET = 27


def score(params):
    return tune.holdout_score(params, X, y, folds, holdout=1, seed=SEED)


reference = tune.run_search("v1_settings", [{}], score, TUNE_DIR, verbose=False)
print(f"v1 settings, same check: ROC-AUC {reference['auc'][0]:.4f} "
      f"with {reference['trees'][0]} trees\n")
grid = tune.run_search("grid", tune.grid_candidates(), score, TUNE_DIR)
rand = tune.run_search("random", tune.random_candidates(BUDGET, seed=SEED), score, TUNE_DIR)
tpe = tune.run_optuna(BUDGET, score, TUNE_DIR, seed=SEED)

In [ ]:
searches = {"grid": grid, "random": rand, "tpe": tpe}
fig, ax = plt.subplots(figsize=(8, 5))
for method, trials in searches.items():
    ax.plot(trials["trial"] + 1, trials["best_so_far"], marker=".", label=method)
ax.axhline(reference["auc"][0], color="grey", linestyle="--", label="v1 settings")
ax.set(xlabel="trial", ylabel="best ROC-AUC so far (fold 1)",
       title=f"Tuning with the same budget ({BUDGET} fits each)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "fig17_tuning_curves.png", dpi=150)
plt.show()

rows = []
for method, trials in searches.items():
    top = trials["auc"].idxmax()
    rows.append({"method": method, "best ROC-AUC": round(trials["auc"].max(), 4),
                 "found at trial": int(top) + 1, "trees": int(trials.loc[top, "trees"]),
                 "minutes": round(trials["seconds"].sum() / 60, 1),
                 "best settings": trials.loc[top, "params"]})
search_summary = pd.DataFrame(rows).set_index("method")
display(search_summary)
pd.concat([reference, grid, rand, tpe], ignore_index=True).to_csv(
    REPORTS / "results_step7_tuning.csv", index=False)

facts["tuning_v1_reference"] = round(float(reference["auc"][0]), 4)
facts["tuning_best"] = search_summary["best ROC-AUC"].to_dict()
facts["tuning_found_at_trial"] = search_summary["found at trial"].to_dict()
facts["tuning_minutes"] = search_summary["minutes"].to_dict()

## The final model

Take the best trial over all three searches and check it properly: full 5-fold CV with the
number of trees it needed in the search, compared with LightGBM v1 (paired bootstrap).
If it does not beat v1, the v1 settings stay.

Then the chosen model is trained once on the **whole train split** and saved. Every later step
(calibration, conformal, explanations, fairness, the app) uses this saved model.

In [ ]:
best = tune.best_trial(grid, rand, tpe)
best_params, best_trees = dict(best["params"]), int(best["trees"])
print(f"best trial: {best['method']} #{best['trial'] + 1}, ROC-AUC on fold 1 {best['auc']:.4f}, "
      f"{best_trees} trees")
print("settings:", best_params)

tuned = run_or_load(lambda: tune.tuned_lightgbm(best_params, best_trees, seed=SEED), X, y, folds,
                    f"step7_lightgbm_tuned_{best['method']}{best['trial'] + 1}", ids=ids, step=7,
                    verbose=True)
tuned.name = "lightgbm_tuned"
vs_v1 = paired_bootstrap(yv, tuned.oof, v1.oof)
display(results_table([tuned, v1]))
print(f"tuned - v1: {vs_v1['difference']:+.4f} "
      f"(95% CI {vs_v1['low']:+.4f} to {vs_v1['high']:+.4f}), significant: {vs_v1['significant']}")

facts["best_trial"] = (best["method"], int(best["trial"]) + 1, round(float(best["auc"]), 4))
facts["best_settings"] = {k: (round(v, 4) if isinstance(v, float) else v)
                          for k, v in best_params.items()}
facts["tuned_cv"] = (round(tuned.summary["roc_auc"], 4),
                     round(tuned.summary["roc_auc_low"], 4), round(tuned.summary["roc_auc_high"], 4))
facts["v1_fixed_cv"] = round(v1.summary["roc_auc"], 4)
facts["tuned_vs_v1"] = (round(vs_v1["difference"], 4), vs_v1["significant"])

In [ ]:
if tuned.summary["roc_auc"] > v1.summary["roc_auc"]:
    chosen, final_params = "tuned", best_params
    final_trees = int(round(best_trees * 1.1))  # 25% more rows than in the search: a few more trees
    cv = tuned.summary
else:
    chosen, final_params, final_trees, cv = "v1 settings (tuning did not beat them)", {}, 500, v1.summary

final_model = tune.tuned_lightgbm(final_params, final_trees, seed=SEED).fit(X, y)
info = {
    "step": 7,
    "chosen": chosen,
    "params": final_params,
    "n_estimators": final_trees,
    "trained_on": f"train split, {len(X):,} rows",
    "cv_roc_auc": round(cv["roc_auc"], 4),
    "cv_roc_auc_95ci": [round(cv["roc_auc_low"], 4), round(cv["roc_auc_high"], 4)],
    "seed": SEED,
}
folder = save_final_model(final_model, X.columns, info)
print(f"final model: {chosen}, {final_trees} trees, saved in {folder}")
facts["final_model"] = (chosen, final_trees, round(cv["roc_auc"], 4))

## Key facts
Send these to Claude to get the filled-in Findings.

In [ ]:
for key, value in facts.items():
    print(f"{key:<28} {value}")

## Findings  ✍️ (fill in after the run, or ask Claude to fill them from the Key facts)

- **A. Fix:** __% of applicants have at least one running previous loan. Their default rate is __% vs __% without one. LightGBM v1 went from __ (old features) to __ (fixed), a change of __ (significant: __).
- **B. Imbalance (ROC-AUC):** none __, class weights __, undersampling __, SMOTE __ (control without SMOTE __). Significant changes vs none: __.
- **B. Probabilities:** the average predicted probability is __ with no fix (actual default rate __), but __ with class weights, __ with undersampling and __ with SMOTE. These methods mostly inflate the probabilities and hurt calibration (Brier __ vs __ with no fix). **Decision:** __.
- **C. Tuning (best ROC-AUC on fold 1 after 27 fits):** grid __, random __, Optuna TPE __; v1 settings __. The best was found at trial __ (grid), __ (random), __ (TPE).
- **C. Check:** the best settings (__ search, __ trees) give __ in full 5-fold CV (95% CI __ to __) vs __ for v1, a change of __ (significant: __).
- **Final model:** __, __ trees, trained on all 184,503 train rows, saved in `models/`.

**Next (Step 8):** neural networks and the from-scratch algorithms.